In [3]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the generated JSON samples
with open("sample_data.json", "r") as f:
    data = json.load(f)

# Load embedding model (local, no API needed)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert each entry to text
documents = []
for entry in data:
    text = (
        f"Name: {entry['name']}, "
        f"Taxonomy: {entry['taxonomy']}, "
        f"Features: {entry['features']}, "
        f"Silviculture: {entry['silviculture']}, "
        f"Regions: {entry['regions']}"
    )
    documents.append(text)

# Generate embeddings
embeddings = model.encode(documents)

# Convert to numpy array
embeddings = np.array(embeddings).astype("float32")

# Create FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# Save FAISS index
faiss.write_index(index, "species_index.faiss")

print("✅ Vector database created with FAISS")


d:\software\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\software\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: ht

✅ Vector database created with FAISS


In [4]:
# Load index back
index = faiss.read_index("species_index.faiss")

# Example query
query = "Fast growing timber tree in India with high drought tolerance"
query_vec = model.encode([query]).astype("float32")

# Search top 3 results
D, I = index.search(query_vec, 3)

print("\n🔎 Top Matches:")
for idx in I[0]:
    print(" -", data[idx]["name"])



🔎 Top Matches:
 - Pinus roxburghii
 - Terminalia arjuna
 - Acacia nilotica
